# 07. Validación y perfiles finales

Dos representaciones, dos preguntas: **cuánto crece** cada artista (intensidad, nivel de log1p centrado, notebooks 04 y 05) y **cómo crece** (forma, tasa de crecimiento centrada, notebook 06). En las dos, la partición elegida es spectral con DTW y k = 3.

Preguntas de este notebook:

1. ¿Los perfiles son estables o dependen de la semilla y de qué artistas entren?
2. ¿Los perfiles son la popularidad disfrazada?
3. ¿Cómo se combinan intensidad y forma?
4. ¿Cómo se describe cada perfil y quién lo representa?

In [ ]:
# PAQUETES

library(data.table)
library(cluster)
library(mclust)
library(ggplot2)

## 1. Configuración y carga

In [ ]:
# ============================================================
# CONFIGURACION
# ============================================================

ARCHIVO_DISTANCIAS_NIVEL <- file.path("chartmetric_pilotos", "distancias_clustering.rds")
ARCHIVO_PARTICIONES_NIVEL <- file.path("chartmetric_pilotos", "particiones_clustering.rds")
ARCHIVO_DISTANCIAS_FORMA <- file.path("chartmetric_pilotos", "distancias_forma_crecimiento.rds")
ARCHIVO_PARTICIONES_FORMA <- file.path("chartmetric_pilotos", "particiones_forma_crecimiento.rds")
FIGURES_DIR <- "figuras"

# Partición elegida en cada representación
K_ELEGIDO <- 3L

# Spectral: misma configuración que los notebooks 05 y 06
VECINO_ESCALA_LOCAL <- 7L
INICIOS_KMEANS <- 25L

# Estabilidad
N_SEMILLAS_KMEANS <- 20L
N_REMUESTREOS <- 50L
PROPORCION_REMUESTREO <- 0.8
N_PANELES_BALANCEADOS <- 10L

SEMILLA_PRIMIGENIA <- 999199L

primos_seis_cifras <- function() {
  limite <- 999999L
  es_primo <- rep(TRUE, limite)
  es_primo[1] <- FALSE
  for (p in 2:floor(sqrt(limite))) {
    if (es_primo[p]) es_primo[seq(p * p, limite, by = p)] <- FALSE
  }
  candidatos <- which(es_primo)
  candidatos[candidatos >= 100000L]
}

PRIMOS_SEIS_CIFRAS <- primos_seis_cifras()

generar_semillas_primas <- function(n, semilla = SEMILLA_PRIMIGENIA) {
  set.seed(semilla)
  orden <- sample(length(PRIMOS_SEIS_CIFRAS))
  PRIMOS_SEIS_CIFRAS[orden[seq_len(n)]]
}

stopifnot(identical(generar_semillas_primas(3), c(742457L, 895957L, 543139L)))

# La primera semilla es la del k-means de las particiones elegidas;
# las siguientes se usan para la estabilidad
SEMILLAS <- generar_semillas_primas(1 + N_SEMILLAS_KMEANS + N_REMUESTREOS + N_PANELES_BALANCEADOS)
SEMILLA_PRINCIPAL <- SEMILLAS[1]
SEMILLAS_KMEANS <- SEMILLAS[1 + seq_len(N_SEMILLAS_KMEANS)]
SEMILLAS_REMUESTREO <- SEMILLAS[1 + N_SEMILLAS_KMEANS + seq_len(N_REMUESTREOS)]
SEMILLAS_BALANCEADO <- SEMILLAS[1 + N_SEMILLAS_KMEANS + N_REMUESTREOS + seq_len(N_PANELES_BALANCEADOS)]

In [ ]:
# ============================================================
# CARGA: matrices DTW, particiones elegidas, artistas y features
# ============================================================

distancias_nivel <- readRDS(ARCHIVO_DISTANCIAS_NIVEL)
distancias_forma <- readRDS(ARCHIVO_DISTANCIAS_FORMA)

MATRICES <- list(intensidad = distancias_nivel$matrices$dtw, forma = distancias_forma$matrices$dtw)
ids_artistas <- rownames(MATRICES$intensidad)

artistas <- copy(distancias_nivel$artistas)[, artist_id := as.character(artist_id)]
artistas <- artistas[match(ids_artistas, artist_id)]

particiones_nivel <- readRDS(ARCHIVO_PARTICIONES_NIVEL)$particiones
particiones_forma <- readRDS(ARCHIVO_PARTICIONES_FORMA)$particiones

asignacion_desde <- function(particiones, algoritmo_elegido) {
  p <- particiones[algoritmo == algoritmo_elegido & k == K_ELEGIDO]
  if ("distancia" %in% names(p)) p <- p[distancia == "dtw"]
  p[match(ids_artistas, artist_id), cluster]
}

PARTICIONES_ELEGIDAS <- list(
  intensidad = asignacion_desde(particiones_nivel, "espectral"),
  forma = asignacion_desde(particiones_forma, "espectral")
)
PARTICIONES_PAM <- list(
  intensidad = asignacion_desde(particiones_nivel, "pam"),
  forma = asignacion_desde(particiones_forma, "pam")
)

archivos_features <- list.files(pattern = "_features_artista[.]csv$")
features <- fread(archivos_features[which.max(file.mtime(archivos_features))])
features[, artist_id := as.character(artist_id)]

# Sanity check: mismos artistas en el mismo orden en todo
stopifnot(
  identical(rownames(MATRICES$forma), ids_artistas),
  !anyNA(unlist(PARTICIONES_ELEGIDAS)), !anyNA(unlist(PARTICIONES_PAM)),
  setequal(features$artist_id, ids_artistas)
)

cat("Artistas:", length(ids_artistas), "\n")
sapply(PARTICIONES_ELEGIDAS, table)

## 2. ¿Los perfiles son estables?

Tres pruebas, todas medidas con el índice de Rand ajustado (ARI: 1 = misma partición, 0 = acuerdo de azar):

- **Semilla:** spectral repetido con 20 semillas del k-means.
- **Remuestreo:** 50 veces, el 80% de los artistas al azar; se recalcula todo (afinidad, autovectores, k-means) y se compara con la partición completa en los artistas que entraron.
- **Algoritmo:** acuerdo entre spectral y PAM.

In [ ]:
# ============================================================
# SPECTRAL (misma implementación que los notebooks 05 y 06)
# ============================================================

agrupar_espectral_matriz <- function(m, k = K_ELEGIDO, semilla = SEMILLA_PRINCIPAL) {
  escala <- apply(m, 1, function(fila) sort(fila)[VECINO_ESCALA_LOCAL + 1])
  afinidad <- exp(-m^2 / outer(escala, escala))
  diag(afinidad) <- 0
  grado <- rowSums(afinidad)
  vectores <- eigen(afinidad / sqrt(outer(grado, grado)), symmetric = TRUE)$vectors[, seq_len(k), drop = FALSE]
  vectores <- vectores / sqrt(rowSums(vectores^2))
  set.seed(semilla)
  kmeans(vectores, centers = k, nstart = INICIOS_KMEANS, iter.max = 100)$cluster
}

# Control: la implementación reproduce las particiones de los notebooks 05 y 06
stopifnot(
  adjustedRandIndex(agrupar_espectral_matriz(MATRICES$intensidad), PARTICIONES_ELEGIDAS$intensidad) > 1 - 1e-9,
  adjustedRandIndex(agrupar_espectral_matriz(MATRICES$forma), PARTICIONES_ELEGIDAS$forma) > 1 - 1e-9
)

In [ ]:
# ============================================================
# ESTABILIDAD: semilla, remuestreo del 80% y algoritmo
# ============================================================

estabilidad <- rbindlist(lapply(names(MATRICES), function(representacion) {
  m <- MATRICES[[representacion]]
  referencia <- PARTICIONES_ELEGIDAS[[representacion]]

  ari_semilla <- vapply(SEMILLAS_KMEANS, function(s)
    adjustedRandIndex(agrupar_espectral_matriz(m, semilla = s), referencia), numeric(1))

  ari_remuestreo <- vapply(SEMILLAS_REMUESTREO, function(s) {
    set.seed(s)
    muestra <- sort(sample(length(ids_artistas), round(PROPORCION_REMUESTREO * length(ids_artistas))))
    adjustedRandIndex(agrupar_espectral_matriz(m[muestra, muestra]), referencia[muestra])
  }, numeric(1))

  data.table(
    representacion = representacion,
    prueba = c("semilla (20)", "remuestreo 80% (50)", "spectral vs PAM"),
    ari_mediano = round(c(median(ari_semilla), median(ari_remuestreo), adjustedRandIndex(referencia, PARTICIONES_PAM[[representacion]])), 3),
    ari_p10 = round(c(quantile(ari_semilla, 0.1), quantile(ari_remuestreo, 0.1), NA), 3)
  )
}))

estabilidad

## 3. ¿Los perfiles son la popularidad disfrazada?

Composición de cada perfil por decil del marco y prueba chi cuadrado. Sensibilidad: 10 paneles balanceados (igual cantidad de artistas por decil, el mínimo disponible), reclusterizados y comparados con la partición completa.

In [ ]:
# ============================================================
# CONTROL POR DECIL DE POPULARIDAD
# ============================================================

MINIMO_POR_DECIL <- min(table(artistas$decil_marco))

control_decil <- rbindlist(lapply(names(PARTICIONES_ELEGIDAS), function(representacion) {
  tabla <- table(cluster = PARTICIONES_ELEGIDAS[[representacion]], decil = artistas$decil_marco)
  prueba <- suppressWarnings(chisq.test(tabla))

  ari_balanceado <- vapply(SEMILLAS_BALANCEADO, function(s) {
    set.seed(s)
    muestra <- sort(unlist(lapply(split(seq_along(ids_artistas), artistas$decil_marco), function(idx) sample(idx, MINIMO_POR_DECIL))))
    adjustedRandIndex(agrupar_espectral_matriz(MATRICES[[representacion]][muestra, muestra]),
                      PARTICIONES_ELEGIDAS[[representacion]][muestra])
  }, numeric(1))

  data.table(
    representacion = representacion,
    chi2_p_valor = signif(prueba$p.value, 2),
    v_de_cramer = round(sqrt(prueba$statistic / (sum(tabla) * (min(dim(tabla)) - 1))), 3),
    panel_balanceado = paste0(MINIMO_POR_DECIL, " por decil"),
    ari_balanceado_mediano = round(median(ari_balanceado), 3)
  )
}))

composicion_decil <- rbindlist(lapply(names(PARTICIONES_ELEGIDAS), function(representacion) {
  data.table(representacion = representacion, cluster = PARTICIONES_ELEGIDAS[[representacion]], decil = artistas$decil_marco)
}))[, .N, by = .(representacion, cluster, decil)][, pct := round(100 * N / sum(N), 1), by = .(representacion, cluster)]

print(control_decil)
cat("\nPorcentaje de cada perfil por decil (1 = más popular):\n")
dcast(composicion_decil, representacion + cluster ~ decil, value.var = "pct")

## 4. Nombres de los perfiles y caracterización

Los nombres se asignan con reglas sobre las features (notebook 03), no a mano:

- **Intensidad:** orden por pendiente anual mediana de las tres audiencias (estable < moderado < fuerte).
- **Forma:** el grupo más grande es "parejo"; de los otros dos, el de mayor salto de suscriptores de YouTube es "salto en YouTube" y el restante, el de menor aceleración, "enfriamiento".

In [ ]:
# ============================================================
# NOMBRES DE LOS PERFILES POR REGLAS SOBRE LAS FEATURES
# ============================================================

AUDIENCIAS <- c("spotify_followers", "instagram_followers", "youtube_channel_subscribers")

features[, `:=`(
  pendiente_audiencias = rowMeans(.SD[, paste0("pendiente_anual__", AUDIENCIAS), with = FALSE]),
  aceleracion_audiencias = rowMeans(.SD[, paste0("aceleracion__", AUDIENCIAS), with = FALSE])
)]
features <- features[match(ids_artistas, artist_id)]
features[, `:=`(cluster_intensidad = PARTICIONES_ELEGIDAS$intensidad, cluster_forma = PARTICIONES_ELEGIDAS$forma)]

orden_intensidad <- features[, .(pendiente = median(pendiente_audiencias)), by = cluster_intensidad][order(pendiente), cluster_intensidad]
nombres_intensidad <- setNames(c("estable", "moderado", "fuerte"), orden_intensidad)

resumen_forma <- features[, .(
  artistas = .N,
  salto_youtube = median(crecimiento_max__youtube_channel_subscribers),
  aceleracion = median(aceleracion_audiencias)
), by = cluster_forma]
cluster_parejo <- resumen_forma[which.max(artistas), cluster_forma]
otros <- resumen_forma[cluster_forma != cluster_parejo]
cluster_salto <- otros[which.max(salto_youtube), cluster_forma]
cluster_enfriamiento <- otros[cluster_forma != cluster_salto, cluster_forma]
nombres_forma <- setNames(c("parejo", "enfriamiento", "salto en YouTube"), c(cluster_parejo, cluster_enfriamiento, cluster_salto))

features[, `:=`(
  perfil_intensidad = factor(nombres_intensidad[as.character(cluster_intensidad)], levels = c("estable", "moderado", "fuerte")),
  perfil_forma = factor(nombres_forma[as.character(cluster_forma)], levels = c("parejo", "enfriamiento", "salto en YouTube"))
)]

# Sanity check: el enfriamiento desacelera más que los parejos
stopifnot(resumen_forma[cluster_forma == cluster_enfriamiento, aceleracion] < resumen_forma[cluster_forma == cluster_parejo, aceleracion])

resumen_forma[, nombre := nombres_forma[as.character(cluster_forma)]][]

In [ ]:
# ============================================================
# CARACTERIZACION: medianas de las features por perfil
# (promedio de las tres audiencias; nivel = tamaño del artista)
# ============================================================

features[, `:=`(
  nivel_audiencias = rowMeans(.SD[, paste0("nivel_log_medio__", AUDIENCIAS), with = FALSE]),
  crecimiento_total_audiencias = rowMeans(.SD[, paste0("crecimiento_total__", AUDIENCIAS), with = FALSE]),
  volatilidad_audiencias = rowMeans(.SD[, paste0("volatilidad_residuo__", AUDIENCIAS), with = FALSE])
)]

caracterizar <- function(columna_perfil) {
  features[
    ,
    .(
      artistas = .N,
      pendiente_anual_pct = round(100 * median(pendiente_audiencias), 1),
      crecimiento_total_pct = round(100 * (exp(median(crecimiento_total_audiencias)) - 1)),
      aceleracion = round(median(aceleracion_audiencias), 4),
      volatilidad = round(median(volatilidad_audiencias), 4),
      nivel_log = round(median(nivel_audiencias), 1),
      decil_mediano = as.numeric(median(decil_marco))
    ),
    keyby = c(perfil = columna_perfil)
  ]
}

cat("Intensidad:\n"); print(caracterizar("perfil_intensidad"))
cat("\nForma:\n"); caracterizar("perfil_forma")

## 5. ¿Cómo se combinan intensidad y forma?

In [ ]:
# ============================================================
# CRUCE INTENSIDAD x FORMA
# ============================================================

cruce <- features[, .N, by = .(perfil_intensidad, perfil_forma)]
cruce[, pct_de_la_intensidad := round(100 * N / sum(N), 1), by = perfil_intensidad]

prueba_cruce <- suppressWarnings(chisq.test(table(features$perfil_intensidad, features$perfil_forma)))
cat("Chi cuadrado intensidad x forma: p =", signif(prueba_cruce$p.value, 2), "\n\n")

print(dcast(cruce, perfil_intensidad ~ perfil_forma, value.var = "N", fill = 0))

grafico_cruce <- ggplot(cruce, aes(x = perfil_forma, y = perfil_intensidad, fill = pct_de_la_intensidad)) +
  geom_tile(colour = "white", linewidth = 1.5) +
  geom_text(aes(label = paste0(N, "\n(", pct_de_la_intensidad, "%)")), size = 4.2, colour = "#1D1E2C") +
  scale_fill_gradientn(colours = c("#F2E8CF", "#F4A259", "#BC4B51"), limits = c(0, 100), name = "% de la fila") +
  labs(
    title = "Cuánto crece y cómo crece",
    subtitle = "Artistas por perfil de intensidad (filas) y de forma (columnas); porcentaje dentro de cada intensidad",
    x = "Forma del crecimiento", y = "Intensidad del crecimiento"
  ) +
  theme_minimal(base_size = 13) +
  theme(panel.grid = element_blank(), plot.background = element_rect(fill = "white", colour = NA))

print(grafico_cruce)

ggsave(file.path(FIGURES_DIR, "perfiles_01_cruce_intensidad_forma.png"), grafico_cruce,
       width = 9, height = 6, dpi = 300, bg = "white")

## 6. ¿Quién representa a cada perfil?

El artista representativo es el medoide: el de menor distancia DTW total al resto de su perfil. Se listan también los dos más cercanos a él.

In [ ]:
# ============================================================
# ARTISTAS REPRESENTATIVOS (medoide y dos vecinos)
# ============================================================

representativos <- function(representacion, columna_perfil) {
  m <- MATRICES[[representacion]]
  rbindlist(lapply(levels(features[[columna_perfil]]), function(perfil) {
    idx <- which(features[[columna_perfil]] == perfil)
    suma <- rowSums(m[idx, idx, drop = FALSE])
    orden <- idx[order(suma)]
    medoide <- orden[1]
    cercanos <- idx[order(m[medoide, idx])][2:3]
    data.table(
      representacion = representacion,
      perfil = perfil,
      artistas = length(idx),
      medoide = artistas$artist_name[medoide],
      mas_cercanos = paste(artistas$artist_name[cercanos], collapse = ", ")
    )
  }))
}

rbind(representativos("intensidad", "perfil_intensidad"), representativos("forma", "perfil_forma"))

In [ ]:
# ============================================================
# EXPORTACION: perfil de cada artista (archivo local, fuera de git)
# ============================================================

perfiles_artistas <- features[, .(artist_id, artist_name, cm_artist_rank, decil_marco, perfil_intensidad, perfil_forma)]

ARCHIVO_PERFILES <- "artistas_ST_perfiles_finales.csv"
fwrite(perfiles_artistas, ARCHIVO_PERFILES)

cat("Exportado:", ARCHIVO_PERFILES, "-", nrow(perfiles_artistas), "artistas\n")

## Lectura

- **Estables.** Intensidad: ARI de 1 frente a la semilla y de 0,92 al remuestrear el 80% de los artistas. Forma: 0,91 al remuestrear; algunas semillas del k-means mueven el grupo chico (p10 de 0,76).
- **No son la popularidad disfrazada.** La composición por decil no difiere entre perfiles (chi cuadrado p = 0,21 y 0,75; V de Cramér menor a 0,1) y los paneles balanceados (107 artistas por decil) reproducen las particiones (ARI 0,96 y 0,99).
- **Intensidad y forma no son independientes** (p < 10⁻¹⁰⁰). Casi todos los estables crecen parejo (579 de 584); el 73% de los de crecimiento fuerte está en enfriamiento (123 de 168). El crecimiento fuerte y sostenido es raro: 20 artistas.
- **Representantes (medoides).** Estable: Nicki Minaj (junto a Rihanna). Moderado: KAYTRANADA. Fuerte: Corey Kent (junto a IVE y Luck Ra). Enfriamiento: Anne Wilson (junto a DannyLux y Rema). El grupo de salto en YouTube (26) queda como posible anomalía de la fuente.
- **Algoritmo.** Spectral y PAM acuerdan en intensidad (ARI 0,69) y no en forma (0,25): PAM aísla los saltos de YouTube. Es la razón de elegir spectral.